# Operaciones avanzadas con DataFrames

## Descripción de las variables

El dataset, obtenido de <a target = "_blank" href="https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ&QO_fu146_anzr=b0-gvzr">este link</a> está compuesto por las siguientes variables referidas siempre al año 2018:

1. **Month** 1-4
2. **DayofMonth** 1-31
3. **DayOfWeek** 1 (Monday) - 7 (Sunday)
4. **FlightDate** fecha del vuelo
5. **Origin** código IATA del aeropuerto de origen
6. **OriginCity** ciudad donde está el aeropuerto de origen
7. **Dest** código IATA del aeropuerto de destino
8. **DestCity** ciudad donde está el aeropuerto de destino  
9. **DepTime** hora real de salida (local, hhmm)
10. **DepDelay** retraso a la salida, en minutos
11. **ArrTime** hora real de llegada (local, hhmm)
12. **ArrDelay** retraso a la llegada, en minutos: se considera que un vuelo ha llegado "on time" si aterrizó menos de 15 minutos más tarde de la hora prevista en el Computerized Reservations Systems (CRS).
13. **Cancelled** si el vuelo fue cancelado (1 = sí, 0 = no)
14. **CancellationCode** razón de cancelación (A = aparato, B = tiempo atmosférico, C = NAS, D = seguridad)
15. **Diverted** si el vuelo ha sido desviado (1 = sí, 0 = no)
16. **ActualElapsedTime** tiempo real invertido en el vuelo
17. **AirTime** en minutos
18. **Distance** en millas
19. **CarrierDelay** en minutos: El retraso del transportista está bajo el control del transportista aéreo. Ejemplos de sucesos que pueden determinar el retraso del transportista son: limpieza de la aeronave, daño de la aeronave, espera de la llegada de los pasajeros o la tripulación de conexión, equipaje, impacto de un pájaro, carga de equipaje, servicio de comidas, computadora, equipo del transportista, problemas legales de la tripulación (descanso del piloto o acompañante) , daños por mercancías peligrosas, inspección de ingeniería, abastecimiento de combustible, pasajeros discapacitados, tripulación retrasada, servicio de inodoros, mantenimiento, ventas excesivas, servicio de agua potable, denegación de viaje a pasajeros en mal estado, proceso de embarque muy lento, equipaje de mano no válido, retrasos de peso y equilibrio.
20. **WeatherDelay** en minutos: causado por condiciones atmosféricas extremas o peligrosas, previstas o que se han manifestado antes del despegue, durante el viaje, o a la llegada.
21. **NASDelay** en minutos: retraso causado por el National Airspace System (NAS) por motivos como condiciones meteorológicas (perjudiciales pero no extremas), operaciones del aeropuerto, mucho tráfico aéreo, problemas con los controladores aéreos, etc.
22. **SecurityDelay** en minutos: causado por la evacuación de una terminal, re-embarque de un avión debido a brechas en la seguridad, fallos en dispositivos del control de seguridad, colas demasiado largas en el control de seguridad, etc.
23. **LateAircraftDelay** en minutos: debido al propio retraso del avión al llegar, problemas para conseguir aterrizar en un aeropuerto a una hora más tardía de la que estaba prevista.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# Leemos los datos y quitamos filas con NA y convertimos a numéricas las columnas inferidas incorrectamente
flightsDF = spark.read\
                 .option("header", "true")\
                 .option("inferSchema", "true")\
                 .csv("abfss://datos@cursosparkucm.dfs.core.windows.net/flights-jan-apr-2018.csv")

# Convertimos a enteros y re-categorizamos ArrDelay en una nueva columna ArrDelayCat
# None (< 15 min), Slight(entre 15 y 60 min), Huge (> 60 min)
# 
cleanFlightsDF = flightsDF.withColumn("ArrDelayCat", 
                                      F.when(F.col("ArrDelay") < 15, "None")\
                                        .when((F.col("ArrDelay") >= 15) & (F.col("ArrDelay") < 60), "Slight")\
                                        .otherwise("Huge"))\
                           .cache()


## Agrupar, agregar y ordenar es una manera sencilla de explorar los datos y comparar lo que ocurre en distintos sectores

Imaginemos que somos los dueños de una web de viajes que rastrea internet en busca de vuelos en agencias y otras páginas, los compara y recomienda el más adecuado para el aeropuerto. Junto con esta recomendación, querríamos dar también información sobre vuelos fiables y no fiables en lo que respecta a la puntualidad. Esto depende de muchos factores, como el origen y destino, duración del vuelo, hora del día, etc. Podemos obtener información útil mediante resúmes de tipo "agrupar y agregar", sin necesidad de un modelo predictivo.

### Agrupación y agregaciones

<div class="alert alert-block alert-success">
<p><b>PREGUNTA</b>: ¿Cuáles son los vuelos (origen, destino) con mayor retraso medio? ¿Cuántos vuelos existen entre cada par de aeropuertos?</p>  Guardar el resultado en la variable average_delay_origin_dest
<p><b>PISTA</b>: Tras hacer las agregaciones para cada pareja "Origin", "Dest" (una agregación para el retraso medio y otra para contar), aplica el método sort(F.col("avgDelay").desc()) para ordenar de forma decreciente por la nueva columna del retraso medio.
</div>

**Nota:** vamos a practicar con la función `F.round(columna, cifras_decimales)` que recibe un objeto columna numérica y el número de cifras decimales a las que queremos redondearlo, y devuelve otro objeto columna numérico redondeado. OJO: el nuevo objeto columna devuelto no conserva el nombre que tenía el objeto original, sino que trae el nombre por defecto "round(col_original, n_cifras)" así que conviene renombrarlo con `alias()`.

In [0]:
average_delay_origin_dest = ...

<div class="alert alert-block alert-success">
    <p><b>PREGUNTA</b>: ¿Es el avión un medio de transporte fiable?</p>
    <p>(a) Mostrar, para cada aeropuerto de destino, el número de vuelos que hay en cada <i>categoría de retraso</i> (variable ArrDelayCat). En lugar de llamar agg(F.count("*")), podemos llamar a la transformación count() sobre el resultado de groupBy(), y creará automáticamente una columna llamada "count" con los conteos para cada grupo. Guardarlo en la variable arr_delay_cat_df.
<p> (b) Ahora agrupa por cada aeropuerto de origen y de destino, y mostrando una columna distinta por cada tipo de retraso, con el recuento del número de vuelos en cada combinación. Guardarlo en la variable arr_delay_cat_pivot_df. PISTA: utilizar la función pivot("colName").</p>

<div class="alert alert-block alert-success">
<p><b>PREGUNTA</b>: ¿Hay relación entre el día de la semana y el retraso a la salida o a la llegada?</p>
    <p> (a) Sin usar la función pivot, calcula el retraso medio a la salida y a la llegada para cada día de la semana y ordena por una de ellas descendentemente.</p>
    <p> (b) Ahora haz lo mismo para cada día pero calculando solamente el retraso medio a la llegada, desagregado por cada aeropuerto de origen y destino, utilizando la función pivot() para generar un DF con tantas columnas como días de la semana, más dos (el origen y destino). </p>
</div>

<div class="alert alert-block alert-info">
<p><b>LA FUNCIÓN PIVOT</b>: Puede ser interesante ver, para cada (Origin, Dest), el retraso promedio por
día de la semana. Si agrupamos por esas tres variables (Origin, Dest, DayOfWeek), nuestro resultado tendría demasiadas filas para ser fácil de visualizar (7 x 1009 ya que hay 1009 combinaciones de (Origin, DayOfWeek)). En cambio, vamos a crear 7 columnas, una por día de la semana, en nuestro resultado DF. Lo haremos utilizando una de las variables de agrupación (DayOfWeek) como <i> variable pivot</i>. Como esta variable tiene 7 valores distintos, se crearán 7 columnas nuevas. De esta manera, visualizaremos toda la información de cada combinación (Origen, Dest) condensada en una fila con 7 columnas con los 7 retrasos promedio correspondientes a ese (Origen, Dest) en cada día de la semana.
</div>

### Operaciones JOIN y de ventana

Estaría bien tener el retraso promedio de una ruta junto a cada vuelo, para que podamos ver qué vuelos tuvieron un retraso que fue superior o inferior al retraso promedio de esa ruta.

<div class="alert alert-block alert-success">
    <b> PREGUNTA </b>:
Usa el averageDelayOriginDestDF creado anteriormente, elimina la columna de conteo y luego únerlo con cleanFlightsDF, utilizando Origin y Dest como columnas de enlace. Finalmente, selecciona solo las columnas Origin, Dest, DayOfWeek, ArrDelay y avg_delay del resultado.
</div>

**PREGUNTA**: ¿qué tipo de JOIN utilizarías? ¿Es relevante en este caso?

In [0]:
average_delay_origin_dest.limit(10).display()

**IMPORTANTE**: cuando el DF resultante del join va a contener columnas con nombres duplicados porque ya existían en ambos DF, esa columna no podrá ser usada en el resultado ya que obtendremos un error de "columna ambigua". Para estos casos existe la operación `alias` sobre el DF (no sobre la columna), de forma que podemos hacer `df.alias("df1").join(otro_df.alias("df2")).select("df1.a", "columna_no_repetida")`.

In [0]:
joined_df = cleanFlightsDF.join(
    average_delay_origin_dest.drop("n"), 
    on = ["Origin", "Dest"],
    how = "left_outer"   # podría ser inner con mismo resultado, porque el DF con el que lo unimos contiene con seguridad todas las combinaciones (Origin, Dest) que hay en cleanFlights
).select("Origin", "Dest", "DayofWeek", "ArrDelay", "avg_delay")

joined_df.display()

In [0]:
#cleanFlightsDF = cleanFlightsDF.alias("cf")

cleanFlightsDF.alias("cf").join(
    average_delay_origin_dest.drop("n").alias("avgdel"), 
    on = (cleanFlightsDF.Origin == average_delay_origin_dest.Origin) &
         (cleanFlightsDF.Dest == average_delay_origin_dest.Dest),
    how = "left_outer"   # podría ser inner con mismo resultado, porque el DF con el que lo unimos contiene con seguridad todas las combinaciones (Origin, Dest) que hay en cleanFlights
).select("avgdel.Origin", "cf.Dest", "avg_delay").limit(10).display()

**PREGUNTA**:repetir la operación utilizando funciones de ventana, sin usar `join`, puesto que al fin y al cabo, todos los datos que estamos usando provienen originalmente de un único DF, y eso nos da una pista que indica que "quizá podríamos evitar el join".

**BONUS**: crear una nueva columna <i>belowAverage</i> que tenga valor True si ArrDelay es menor que el avgDelay de esa ruta, y False en caso contrario. No utilizar la función when() sino el operador de comparación directamente entre columnas, la cual devolverá una columna booleana.

In [0]:
from pyspark.sql import Window

w = ...

df_with_avg = cleanFlightsDF...

df_with_avg.limit(100).display()

### Funciones de agregación para traer, en cada grupo, todos los valores de cierta columna que hay en ese grupo. Esto genera una columna de tipo lista o de tipo conjunto

#### Funciones F.collect_list() y F.collect_set(), que pueden utilizarse dentro de agg() en groupBy().agg() y también en una ventana, como ocurre con cualquier función de agregación

La función collect_list() devuelve una nueva columna de tipo lista que puede tener elementos repetidos. La función collect_set devuelve una columna de tipo conjunto donde no hay repetidos.

#### Funciones para manejar columnas de tipo vector: todas las que empiezan por array_ [aquí](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
todasFechasDF = flightsDF.groupBy("Origin", "Dest")\
                    .agg(F.collect_list("FlightDate").alias("todas_fechas"),
                         F.collect_set("FlightDate").alias("fechas_sin_repetidos")
                    )\
                    .withColumn("primer_elemento", F.element_at("todas_fechas", 1)) # primer elemento de cada array

# mostramos 3 filas y solo algunas columnas para que no sea muy largo
todasFechasDF.select("Origin", "Dest", "todas_fechas", "fechas_sin_repetidos", "primer_elemento").limit(3).display()

### Funciones para crear una columna de tipo estructura

Una columna de tipo estructura representa un tipo de dato creado por el usuario, que puede ser "plano", es decir, ser simplemente una tupla, o puede ser "jerárquico" porque dentro de la estructura existan campos que son, a su vez, otras estructuras. El caso más simple es crear una tupla con los valores de varias columnas, fila a fila. Es posible "ordenar" tuplas, porque ordena en base al primer elemento de cada tupla (el más a la izquierda), y si empatan entonces se fija en el segundo, y así sucesivamente.

Esto es útil cuando queremos ordenar dentro de un grupo en base a cierta columna, y después queremos ver el valor concreto de otra columna diferente que estaba en la misma fila, emparejado con el valor máximo o mínimo (es decir, cuando buscamos máximo o mínimo pero *no queremos perder la correspondencia* de ese valor con otros valores de su fila).

Se puede acceder a cada uno de los campos de una tupla con el operador `.` (punto)

**Ejemplo**: para cada par de aeropuertos origen y destino, encontrar la fecha en la que tuvo lugar el vuelo con más retraso a la llegada. Como estamos creando una columna de tipo estructura con dos campos, de los cuales el primero es el ArrDelay, al buscar dentro de cada grupo el máximo de dicha columna de tipo estructura, se fijará justamente en el ArrDelay.

In [0]:
from pyspark.sql import Window

fechaMaxDelayDF = flightsDF\
                    .withColumn("estructura", F.struct("ArrDelay", "FlightDate"))\
                    .groupBy("Origin", "Dest")\
                    .agg(F.max("estructura").alias("max_struct"))\
                    .withColumn("fecha_max_retraso", F.col("max_struct.FlightDate"))

fechaMaxDelayDF.display()

<div class="alert alert-block alert-success">
<b> PREGUNTA </b>: Vamos a construir otro DF con información sobre los aeropuertos (en una situación real, tendríamos otra tabla en la base de datos como la tabla de la entidad Aeropuerto). Sin embargo, solo tenemos información sobre algunos aeropuertos. Nos gustaría agregar esta información a cleanFlightsDF como nuevas columnas, teniendo en cuenta que queremos que la información del aeropuerto coincida con el aeropuerto de origen de flightsDF. Utilizar la operación de unión adecuada para asegurarse de que no se perderá ninguna de las filas existentes de cleanFlightsDF después de la unión.
</div>

In [0]:
from pyspark.sql import types as T

airportsDF = spark.createDataFrame([
        ("JFK", "John F. Kennedy International Airport", 1948),
        ("LIT", "Little Rock National Airport", 1931),
        ("SEA", "Seattle-Tacoma International Airport", 1949),
    ], 
    ["IATA", "FullName", "Year"],    # se podrían indicar solamente los nombres de columna y dejar que Spark infiera el tipo de dato
    
    #schema="IATA string, FullName string, Year int"
    # schema=T.StructType([
    #     T.StructField("IATA", T.StringType(), nullable=True),
    #     T.StructField("FullName", T.StringType(), nullable=True),
    #     T.StructField("Year", T.IntegerType(), nullable=True)
    # ])
    )

airportsDF.show(truncate=False)

## User-defined functions (UDFs)

Vamos a construir un UDF para convertir millas a kilómetros. Ten en cuenta que esto podría hacerse fácilmente multiplicando directamente la columna de millas por 1.6 (y sería mucho más eficiente), ya que Spark permite el producto entre una columna y un número. En todos los casos en los que Spark proporciona funciones integradas para realizar una tarea (como esta), debes usar esas funciones y no una UDF. Las UDF deben emplearse solo cuando no hay otra opción.

La razón es que las funciones integradas de Spark están optimizadas y Catalyst, el optimizador automático de código integrado en Spark, puede optimizarlo aún más. Sin embargo, las UDF son una caja negra para Catalyst y su contenido no se optimizará, y por lo tanto, generalmente son mucho más lentas.

**¿Cuándo sí son una opción muy adecuada?**

Cuando pretendemos utilizar un paquete de Python que no es distribuido, y que implementa una funcionalidad completa que necesitamos ir aplicando a cada fila de nuestro DF, bien a cada celda de una columna en concreto, o bien para un cálculo que realizamos fila a fila, involucrando en cada fila a varias columnas en concreto.

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# Primer paso: crear una función de Python que reciba UN número y lo multiplique por 1.6
def milesToKm(miles):
    return miles*1.6

# Vamos a probarla
print(milesToKm(5)) # 5 millas a km: 8 km

# Segundo paso: crear un objeto UDF que envuelva a nuestra función. 
# Hay que especificar el tipo de dato que devuelve nuestra función
udfMilesToKm = F.udf(milesToKm, DoubleType())

# Con esto, Spark será capaz de llamar a nuestra función milesToKm sobre cada uno de los valores de una columna numérica.
# Spark enviará el código de nuestra función a los executors a través de la red, y cada executor la ejecutará sobre las
# particiones (una por una) que estén en ese executor

# Tercer paso: vamos a probar la UDF añadiendo una nueva columna con el resultado de la conversión
flightsWithKm = cleanFlightsDF.withColumn("DistKm", udfMilesToKm(F.col("Distance"))

# Esto sería una manera mucho mejor de expresar el mismo cálculo
# flightsWithKm = cleanFlightsDF.withColumn("DistKm", 1.6 * F.col("Distance"))


flightsWithKm.select("Origin", "Dest", "Distance", "DistKM")\
             .distinct()\
             .show(5)

<div class="alert alert-block alert-info">
<p><b>BONUS</b>: Crea tu propia UDF que convierta DayOfWeek en una cadena.
Puedes hacerlo creando una función de Python que reciba un número entero y devuelva el día de la semana,
simplemente leyendo desde un vector de cadenas de longitud 7 el valor en la posición indicada por el argumento entero. Para la UDF, recuerda que tu función devuelve un StringType(). Finalmente, prueba tu UDF creando una nueva columna "DayOfWeekString".
</div>

In [0]:
cleanFlightsDF.printSchema()

In [0]:
from pyspark.sql.types import StringType

from pyspark.sql import types as T

# Primer paso: creamos una función de python que convierte un número entero en el día de la semana como cadena
def dayOfWeekToString(dayInteger):
    # En nuestros datos Monday es 1 pero las listas de python empiezan en el 0 y 
    # queremos usar el dayInteger como índice del vector
    daysOfWeek = ["", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    return daysOfWeek[dayInteger]
    
# Segundo paso: ajustamos nuestra función con un Spark UDF para que Spark pueda invocarlo en cada valor de una columna completa
# De esta manera, Spark puede enviar nuestra función a los ejecutores, que eventualmente ejecutarán la función en las particiones
# de los datos que tiene cada ejecutor
dayOfWeekStringUDF = F.udf(dayOfWeekToString, T.StringType())

# Tercer paso: intentemos nuestro UDF agregando una nueva columna que resulta de transformar (a través del UDF) el
# columna existente DayOfWeek

# flightsWithDayOfWeekStr = cleanFlightsDF.withColumn("DayOfWeekString", 
#                                                     dayOfWeekStringUDF(F.col("DayOfWeek")))

resultado_df = cleanFlightsDF.select("Origin", "Dest", "DayOfWeek",
                               dayOfWeekStringUDF(F.col("DayOfWeek")).alias("DoWString"))\
                       .distinct()

resultado_df.show()

## UDF con argumentos que no son columnas (usando currificación)

Para crear una UDF parametrizable, donde hay parámetros que no son objetos Column, podemos crear dos funciones anidadas, donde el resultado que devuelve la función más externa, cuyos argumentos son de tipo no-columna, es en realidad un objeto UDF. Como la función de dicha UDF sí accede a esos parámetros externos, la UDF se crear en realidad a partir de la *clausura* de la función interna, que está compuesta por la propia función más los objetos externos a los que accede. Es decir, estaremos creando una UDF a partir de la "función ya personalizada" para los valores concretos de los parámetros que le queramos pasar.

In [0]:
from pyspark.sql.types import DoubleType

def udf_currificada(param_config):
    """
    El parámetro param_config configura si la función se comporta como suma o como producto. 
    Este parámetro no proviene de una columna del DF sino que lo indicamos nosotros de antemano
    """
    def funcion_interna(x1, x2):
        if param_config == "suma":
            return x1+x2
        elif param_config == "producto":
            return x1*x2

    # Envolvemos como UDF el objeto función junto al parámetro que hemos pasado. Aunque param_config
    # es externo a la función "funcion_interna", dicho parámetro es necesario para ejecutar esa función y por
    # tanto, forma parte de la "clausura" de la función interna: es un "dato adjunto" de la función interna
    return F.udf(funcion_interna, DoubleType())

udf_para_usar = udf_currificada("producto")

cleanFlightsDF.where("ArrDelay is not null and DepDelay is not null")\
              .select(udf_currificada("producto")(F.col("ArrDelay"), F.col("DepDelay")).alias("suma_o_producto"))\
              .show()

## UDF vectorizadas (Pandas UDF)

Todas estas funciones están exentas de traducción de objetos del lenguaje Scala/Java representados en la RAM, y objetos del lenguaje Python, ya que las PandasUDF utilizan siempre pyarrow por debajo.

#### 1. Funciones de pd.Series en pd.Series

La función del usuario será invocada dentro de cada partición, tantas veces como "trozos" tenga la partición, donde Spark compone para cada subconjunto de filas un objeto Series con los trozos correspondientes de columna. No podemos hacer ninguna suposición sobre qué filas vendrán en cada trozo, ya que los trozos no representan ninguna agrupación en base a ninguna columna. El número de filas de esos trozos es generalmente menor que las filas de una partición, y se configura en el parámetro `spark.sql.execution.arrow.maxRecordsPerBatch` cuyo valor por defecto es 10,000.

In [0]:
import pandas as pd
from pyspark.sql import functions as F

@F.pandas_udf("double")   # ponemos el tipo de dato de la serie devuelta
def velocidad(air_time: pd.Series, distance: pd.Series) -> pd.Series:
    v = distance / air_time
    return v

flightsDF.withColumn("velocidad", velocidad("AirTime", "Distance")).limit(100).display()


#### 2. Funciones de una pd.Series en un valor (útiles para agregaciones por grupos)



In [0]:
@F.pandas_udf("double")
def mediana(arr_delay: pd.Series) -> float:
    return arr_delay.median()

agg_df = flightsDF.groupBy("Origin").agg(mediana("Distance").alias("mediana_dist"))
agg_df.display()

#### 3. mapInPandas()

Podemos controlar el tamaño de cada batch (número de filas que se devuelven en cada DataFrame de Pandas) con el parámetro `spark.sql.execution.arrow.maxRecordsPerBatch`. De nuevo no podemos hacer ninguna suposición sobre qué filas concretas va a tener cada "batch" (cada pd.DataFrame que nos va dando el iterador del argumento de la función).

In [0]:
# Muy útiles para hacer análisis complejos por grupos o por trozos de DF
import copy
import pandas as pd
from pyspark.sql import types as T
from typing import Iterator

schema = flightsDF.schema
schema_devuelto: T.StructType = copy.deepcopy(schema)
schema_devuelto.add('Speed', T.DoubleType(), True)

def velocidad_df(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    for pdf in iterator:
        yield pdf.assign(Speed=pdf["Distance"] / pdf["AirTime"])

speed_df = flightsDF.mapInPandas(velocidad_df, schema=schema_devuelto)
speed_df.display()

#### 4. groupBy().applyInPandas()

El pd.DataFrame que recibimos como argumento representa todas las filas de un grupo completo, por lo que debemos asegurarnos de que las filas de un grupo siempre caben en la memoria RAM de cualquier executor, sea cual sea el grupo. Spark decide a qué executor envía las filas de cada grupo, para ejecutar allí la función del usuario. Se invocará la función tantas veces como grupos existan, y Spark distribuirá los grupos entre los distintos executors, de forma equilibrada en lo posible. El esquema devuelto puede ser arbitrario, y puede devolverse un número arbitrario de filas en cada grupo. No está sometido a la restricción habitual de `groupBy().agg()` donde el resultado tiene tantas filas como combinaciones existan de los valores de las variables agrupadoras: aquí no se cumple eso, y el número de filas calculadas como resultado puede ser mayor o menor que el número de filas que tenía originalmente el grupo. Además, se incluyen todas las columnas que tuviese el DF original y/o que hayamos querido calcular nuevas en nuestra función.

In [0]:
from pyspark.sql.types import DoubleType, StructType, StructField, StringType, DoubleType
import pandas as pd

def utiliza_grupo(grupo_df: pd.DataFrame) -> pd.DataFrame:
    grupo_df["mediana"] = grupo_df["ArrDelay"].median()
    return grupo_df


resultado = flightsDF\
             .where("ArrDelay is not null")\
             .withColumn("ArrDelay", F.col("ArrDelay").cast(DoubleType()))\
             .select("Origin", "Dest", "ArrDelay")\
             .groupBy("Origin")\
             .applyInPandas(utiliza_grupo, schema="Origin string, Dest string, ArrDelay float, mediana float")

resultado.limit(100).display()